# Complete Workflow Example

This notebook demonstrates the RIS-first workflow for literature-derived marine arsenic measurement reconstruction. It is designed as an executable guide, but expensive or long-running steps are controlled by switches.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path(r"D:\\wuqiang\\qiang\\arsenic\\arsenic_reconstruction_release")
SRC = PROJECT_ROOT / "src"
for path in [PROJECT_ROOT, SRC]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

# Safe defaults. Turn these on only when you want to run the full workflow.
RUN_REAL_QWEN = False
RUN_PDF_PARSE = False
RUN_REVIEW_STAGES = False

print(PROJECT_ROOT)

## 1. Load Configuration

Extraction run count and thinking mode are configured in `config/prompts/extraction/extraction_v1.yaml`. Review-stage folder names are configured in `config/review_stages.yaml`.

In [ ]:
from arsenic_workflow.qwen_extraction import load_extraction_prompt_spec
from arsenic_workflow.review_workspace import load_review_config

extraction_prompt = load_extraction_prompt_spec(PROJECT_ROOT)
review_config = load_review_config(PROJECT_ROOT)

print("Extraction prompt:", extraction_prompt)
print("Review root:", review_config["root"])
print("Stage directories:")
for key, value in review_config["stage_dirs"].items():
    print(f"  {key}: {value}")

## 2. RIS Ingestion and Abstract Screening

The example RIS file is `data/ris/example.ris`. It may contain abstracts exported by WoS, EndNote, Zotero, or another RIS exporter. The parser uses `rispy` rather than custom line parsing.

In [ ]:
import pandas as pd
from arsenic_workflow.ris_ingest import (
    DEFAULT_SCREENING_PROMPT,
    initialize_literature_index_from_ris,
    load_literature_index,
    run_screening,
    write_literature_index,
)

index_path = initialize_literature_index_from_ris(PROJECT_ROOT)
index = load_literature_index(PROJECT_ROOT)

if RUN_REAL_QWEN:
    index = run_screening(PROJECT_ROOT, index, prompt_path=DEFAULT_SCREENING_PROMPT)
    write_literature_index(PROJECT_ROOT, index.to_dict(orient="records"))

index[["source_id", "doi", "title", "preliminary_screened", "screening_decision", "source_file_exists"]]

## 3. Parse PDF Sources into Text Layers

Each article source PDF is parsed into page text, page images, section files, and four text layers:

- `full_text.txt`
- `no_ref_text.txt`
- `ocrl_text.txt`
- `manual_text.txt`

The priority order is `full_text < no_ref_text < ocrl_text < manual_text`. Empty files are skipped when selecting text for Qwen extraction.

In [ ]:
from arsenic_workflow.source_parse import parse_all_article_pdfs

if RUN_PDF_PARSE:
    # Set ocr_on_abnormal=True to enable DashScope OCR fallback when native text is abnormal.
    parse_results = parse_all_article_pdfs(PROJECT_ROOT, dpi=180, overwrite=True, ocr_on_abnormal=True)
else:
    parse_results = "Skipped. Set RUN_PDF_PARSE = True to parse PDFs."

parse_results

In [ ]:
def read_text_priority(article_id: str) -> dict:
    path = PROJECT_ROOT / "data" / "articles" / article_id / "source" / "extracted_text" / "text_priority_manifest.json"
    return json.loads(path.read_text(encoding="utf-8"))

for article_id in ["test_1", "ssrn_4476876"]:
    manifest = PROJECT_ROOT / "data" / "articles" / article_id / "source" / "extracted_text" / "text_priority_manifest.json"
    if manifest.exists():
        selected = read_text_priority(article_id)["selected_text"]
        print(article_id, "->", selected)

## 4. Two-Pass Qwen Extraction and Concentration Agreement

The number of extraction runs is read from `extraction_v1.yaml`. For each article, the workflow selects the highest-priority non-empty text file, sends it to Qwen for repeated extraction, and keeps only records whose concentration agrees across at least two runs.

In [ ]:
from scripts.run_extraction_consensus import run_article, article_dirs

if RUN_REAL_QWEN:
    extraction_results = []
    for article_dir in article_dirs(PROJECT_ROOT, article_ids=["test_1"]):
        extraction_results.append(
            run_article(PROJECT_ROOT, article_dir, extraction_prompt, runs=int(extraction_prompt["runs"]))
        )
else:
    extraction_results = "Skipped. Set RUN_REAL_QWEN = True to call Qwen."

extraction_results

In [ ]:
def read_consensus_records(article_id: str, version: str = "v1") -> pd.DataFrame:
    path = PROJECT_ROOT / "data" / "articles" / article_id / "source" / version / "final" / "concentration_consensus_records.csv"
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

test_consensus = read_consensus_records("test_1")
test_consensus

## 5. Manual-Review Workspace

From this point onward, all non-article CSVs are stored under `data/review_workspace`. Every stage has two folders:

- `raw_csv`: machine-generated output
- `manual_corrected_csv`: reviewer-editable input for the next stage

The next stage always reads from the previous stage's `manual_corrected_csv` folder.

In [ ]:
from arsenic_workflow.review_workspace import run_review_stage, run_review_workflow

if RUN_REVIEW_STAGES:
    review_results = run_review_workflow(PROJECT_ROOT)
else:
    review_results = "Skipped. Set RUN_REVIEW_STAGES = True to create review-stage CSVs."

review_results

In [ ]:
def list_review_workspace() -> pd.DataFrame:
    root = PROJECT_ROOT / review_config["root"]
    rows = []
    for stage_key, stage_name in review_config["stage_dirs"].items():
        stage_dir = root / stage_name
        rows.append({
            "stage_key": stage_key,
            "stage_dir": str(stage_dir),
            "raw_csv_exists": (stage_dir / "raw_csv").exists(),
            "manual_corrected_csv_exists": (stage_dir / "manual_corrected_csv").exists(),
        })
    return pd.DataFrame(rows)

list_review_workspace()

## 6. Plotting and Modeling Helper Functions

These functions are kept in the notebook for quick review plots and lightweight model checks after the staged CSVs have been generated and manually corrected.

In [ ]:
import matplotlib.pyplot as plt

def read_manual_stage(stage_key: str, filename: str) -> pd.DataFrame:
    stage_name = review_config["stage_dirs"][stage_key]
    path = PROJECT_ROOT / review_config["root"] / stage_name / "manual_corrected_csv" / filename
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

def plot_stage_record_counts() -> None:
    rows = []
    for stage_key, stage_name in review_config["stage_dirs"].items():
        manual_dir = PROJECT_ROOT / review_config["root"] / stage_name / "manual_corrected_csv"
        csv_files = sorted(manual_dir.glob("*.csv")) if manual_dir.exists() else []
        count = 0
        for csv_file in csv_files:
            try:
                count += len(pd.read_csv(csv_file))
            except Exception:
                pass
        rows.append({"stage": stage_name, "records": count})
    df = pd.DataFrame(rows)
    ax = df.plot.bar(x="stage", y="records", legend=False, figsize=(10, 4))
    ax.set_ylabel("Record count")
    ax.set_xlabel("")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()

def plot_concentration_distribution(df: pd.DataFrame, column: str = "arsenic_mg_per_kg_ww") -> None:
    if df.empty or column not in df:
        print(f"Missing column or empty data: {column}")
        return
    values = pd.to_numeric(df[column], errors="coerce").dropna()
    ax = values.plot.hist(bins=20, figsize=(7, 4))
    ax.set_xlabel(column)
    ax.set_ylabel("Frequency")
    plt.tight_layout()

def plot_coordinate_scatter(df: pd.DataFrame, lat_col: str = "latitude_for_environment", lon_col: str = "longitude_for_environment") -> None:
    if df.empty or lat_col not in df or lon_col not in df:
        print("Missing coordinate columns or empty data.")
        return
    lat = pd.to_numeric(df[lat_col], errors="coerce")
    lon = pd.to_numeric(df[lon_col], errors="coerce")
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(lon, lat, s=24, alpha=0.7)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("Reviewed sample coordinates")
    plt.tight_layout()

def fit_linear_model(df: pd.DataFrame, response: str, predictors: list[str]):
    import statsmodels.api as sm
    available = [col for col in predictors if col in df]
    if response not in df or not available:
        raise ValueError("Missing response or predictor columns.")
    model_df = df[[response] + available].apply(pd.to_numeric, errors="coerce").dropna()
    if len(model_df) < len(available) + 2:
        raise ValueError("Not enough complete rows for this model.")
    X = sm.add_constant(model_df[available])
    y = model_df[response]
    return sm.OLS(y, X).fit()

print("Plotting and modeling helpers loaded.")

## 7. Example Review Usage

After `RUN_REVIEW_STAGES = True`, manually inspect and edit the CSVs in each `manual_corrected_csv` folder before running the next stage. The commands below show individual-stage execution order.

In [ ]:
stage_order = [
    "extraction_aggregation",
    "measurement_harmonization",
    "worms_taxonomy_matching",
    "geographic_review",
    "environment_matching",
    "final_output",
]

for stage in stage_order:
    print(f"python scripts/run_review_workspace.py {PROJECT_ROOT} --stage {stage}")